[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DeutscheAktuarvereinigung/.github/blob/main/traffic/traffic_analysis.ipynb)
# Traffic Analysis – DeutscheAktuarvereinigung

Dieses Notebook lädt die aktuellste GitHub-Traffic-CSV aus Google Drive und analysiert sie interaktiv mit Altair.

In [20]:
# Setup: Bibliotheken importieren
import pandas as pd
import altair as alt
import glob
import os
from urllib.error import HTTPError # Import HTTPError

In [9]:
today = pd.Timestamp.today()
print(f"{today:%Y-%m-%d}")
OUTPUT_CSV_PATH = f"https://raw.githubusercontent.com/DeutscheAktuarvereinigung/.github/main/traffic/github_traffic_dav_{today:%Y-%m-%d}.csv"

# CSV laden
try:
    df_from_csv = pd.read_csv(OUTPUT_CSV_PATH, sep=';')
    # Convert 'Datum' column to datetime objects
    df_from_csv['Datum'] = pd.to_datetime(df_from_csv['Datum'])
    print(f'CSV loaded successfully from: {OUTPUT_CSV_PATH}')
except FileNotFoundError:
    print(f'Error: The file {OUTPUT_CSV_PATH} was not found. Please ensure the data collection cell was executed and the file was saved.')
    raise

# Re-create df_clean for charting purposes, using the data loaded from CSV
df_clean_from_csv = df_from_csv[((df_from_csv['Datum'] > (df_from_csv['Datum'].max() - pd.Timedelta(days=14))) &
                                 (df_from_csv['Repository'] != '.github'))]

# Define metrics list
metrics = ['Views', 'Clones', 'Unique Views', 'Unique Clones']

display(df_clean_from_csv.head())
print('Metrics for charting:', metrics)

2026-02-26
CSV loaded successfully from: https://raw.githubusercontent.com/DeutscheAktuarvereinigung/.github/main/traffic/github_traffic_dav_2026-02-26.csv


,Repository,Datum,Views,Clones,Unique Views,Unique Clones
0,-2025_CADS_Immersion_Best_Notebooks,2026-02-24,6,0,2,0
1,-2025_CADS_Immersion_Best_Notebooks,2026-02-23,0,0,0,0
2,-2025_CADS_Immersion_Best_Notebooks,2026-02-22,0,0,0,0
3,-2025_CADS_Immersion_Best_Notebooks,2026-02-21,0,0,0,0
4,-2025_CADS_Immersion_Best_Notebooks,2026-02-20,0,0,0,0


Metrics for charting: ['Views', 'Clones', 'Unique Views', 'Unique Clones']


In [10]:
import altair as alt

# Create a selection for the metric dropdown
metric_selection = alt.selection_point(
    fields=['Metric'],
    bind=alt.binding_select(options=metrics, name='Select Metric '),
    value=metrics[0],
    name='metric_param'
)

# Create a multi-selection for repositories
repository_selection = alt.selection_point(
    fields=['Repository'],
    bind=alt.binding_select(options=df_clean_from_csv['Repository'].unique().tolist(), name='Select Repository '),
    name='repo_select'
)

# Create the base chart
base = alt.Chart(df_clean_from_csv).properties(
    title='GitHub Traffic Trends by Repository'
).add_params(
    metric_selection,
    repository_selection
).transform_fold(
    metrics,
    as_=['Metric', 'Value']
).transform_filter(
    metric_selection
).transform_filter(
    repository_selection
)

# Create the line chart
chart = base.mark_line(point=True).encode(
    x=alt.X('Datum:T', title='Date'),
    y=alt.Y('Value:Q', title='Metric Value', stack=None),
    color=alt.Color('Repository:N', title='Repository'),
    tooltip=[
        alt.Tooltip('Repository:N'),
        alt.Tooltip('Datum:T', title='Date'),
        alt.Tooltip('Value:Q', title='Metric Count', format='.0f')
    ]
).interactive()

chart

alt.Chart(...)

In [21]:
last_week = today - pd.Timedelta(days=7)
print(last_week)

OUTPUT_CSV_PATH = f"https://raw.githubusercontent.com/DeutscheAktuarvereinigung/.github/main/traffic/github_traffic_dav_{last_week:%Y-%m-%d}.csv"

# CSV laden
try:
    df_from_csv = pd.read_csv(OUTPUT_CSV_PATH, sep=';')
    # Convert 'Datum' column to datetime objects
    df_from_csv['Datum'] = pd.to_datetime(df_from_csv['Datum'])
    print(f'CSV loaded successfully from: {OUTPUT_CSV_PATH}')
except HTTPError:
    print(f'Error: The file {OUTPUT_CSV_PATH} was not found. Please ensure the data collection cell was executed and the file was saved.')
    # raise

# Re-create df_clean for charting purposes, using the data loaded from CSV
df_clean_last_week = df_from_csv[((df_from_csv['Datum'] > (df_from_csv['Datum'].max() - pd.Timedelta(days=14))) &
                                 (df_from_csv['Repository'] != '.github'))]
df_clean_from_csv = pd.concat([df_clean_from_csv, df_clean_last_week]).drop_duplicates().reset_index(drop=True)




2026-02-19 15:41:13.235677
Error: The file https://raw.githubusercontent.com/DeutscheAktuarvereinigung/.github/main/traffic/github_traffic_dav_2026-02-19.csv was not found. Please ensure the data collection cell was executed and the file was saved.


In [23]:
# Ensure 'Datum' is datetime type
df_clean_from_csv['Datum'] = pd.to_datetime(df_clean_from_csv['Datum'])

# Extract week from 'Datum'
df_clean_from_csv['Week'] = df_clean_from_csv['Datum'].dt.to_period('W').astype(str)

# Create the crosstab for unique views
weekly_unique_views_crosstab = pd.pivot_table(
    df_clean_from_csv,
    values='Unique Views',
    index='Repository',
    columns='Week',
    aggfunc='sum' # Sum unique views per week per repository
)

print("Wöchentliche eindeutige Aufrufe (Unique Views) nach Repository:")
display(weekly_unique_views_crosstab)

Wöchentliche eindeutige Aufrufe (Unique Views) nach Repository:


Week,2026-02-09/2026-02-15,2026-02-16/2026-02-22,2026-02-23/2026-03-01
Repository,,,
-2025_CADS_Immersion_Best_Notebooks,2,6,2
2024_CADS_Immersion_Best_Notebooks,4,7,6
2025_CADS_Completion_Best_Notebooks,2,2,3
ADS_Use_Cases,0,0,1
Data-Science-Challenge2021_Explainable-Machine-Learning,1,0,1
Data_Science_Challenge_2020_Berufsunfaehigkeit,0,4,1
Data_Science_Challenge_2020_Betrugserkennung,1,2,1
Data_Science_Challenge_2022_Python-Notebook_zur_Erstellung_von_Schadenhaeufigkeitsmodellen,3,0,1
Deriving-NHANES-data-set-CDC-for-mortality-analysis,1,1,1
